In [3]:
import numpy as np
np.set_printoptions(precision=5, suppress=True)


def compute_svd_from_scratch(A: np.ndarray) -> tuple:
    m, n = A.shape

    
    AtA = A.T @ A

    print("Step 1: A^T @ A =")
    print(AtA)
    print()

    eigenvalues, eigenvectors = np.linalg.eig(AtA)
    eigenvalues = np.real(eigenvalues)
    eigenvectors = np.real(eigenvectors)

    
    sort_indices = np.argsort(eigenvalues)[::-1]

    eigenvalues_sorted = eigenvalues[sort_indices]
    eigenvectors_sorted = eigenvectors[:, sort_indices]

    print("Step 3: Sorted eigenvalues:", eigenvalues_sorted)
    print()

   
    singular_values = np.sqrt(np.clip(eigenvalues_sorted, 0, None))

    print("Step 4: Singular values:", singular_values)
    print()

    
    V = np.zeros_like(eigenvectors_sorted)
    for i in range(n):
        norm = np.linalg.norm(eigenvectors_sorted[:, i])
        if norm > 0:
            V[:, i] = eigenvectors_sorted[:, i] / norm

    Vt = V.T

    print("Step 5: V^T =")
    print(Vt)
    print()

    
    U = np.zeros((m, m))
    for i in range(min(m, n)):
        if singular_values[i] > 1e-10:
            U[:, i] = (A @ V[:, i]) / singular_values[i]

    print("Step 6: U =")
    print(U)
    print()

    
    Sigma = np.zeros((m, n))
    for i in range(min(m, n)):
        Sigma[i, i] = singular_values[i]

    print("Step 7: Σ =")
    print(Sigma)
    print()

    return U, Sigma, Vt


def verify_svd(A, U, Sigma, Vt):
    
    reconstructed = U @ Sigma @ Vt

    print("Reconstructed matrix:")
    print(reconstructed)
    print()

    error = np.max(np.abs(A - reconstructed))
    print(f"Max reconstruction error: {error:.6f}")

    return error < 1e-5


def truncated_svd(U, Sigma, Vt, k):
    
    U_k = U[:, :k]
    Sigma_k = Sigma[:k, :k]
    Vt_k = Vt[:k, :]

    return U_k, Sigma_k, Vt_k


def predict_rating_for_new_user(Vt_k, known_ratings, target_item):
    observed_items = list(known_ratings.keys())
    observed_ratings = np.array([known_ratings[i] for i in observed_items])

   
    Vt_observed = Vt_k[:, observed_items]

    print("Observed V^T submatrix:")
    print(Vt_observed)
    print()

    
    x, _, _, _ = np.linalg.lstsq(Vt_observed.T, observed_ratings, rcond=None)

    print("User latent coordinates:", x)
    print()

    
    predicted_rating = x @ Vt_k[:, target_item]

    return predicted_rating



if __name__ == "__main__":
    A = np.array([
        [5, 5, 0],
        [4, 5, 1],
        [5, 4, 1]
    ], dtype=float)

    U, Sigma, Vt = compute_svd_from_scratch(A)
    print("SVD valid:", verify_svd(A, U, Sigma, Vt))

    k = 2
    U_k, Sigma_k, Vt_k = truncated_svd(U, Sigma, Vt, k)

    pred1 = predict_rating_for_new_user(Vt_k, {0: 5, 2: 4}, 1)
    print("Prediction [5, ?, 4]:", pred1)

    pred2 = predict_rating_for_new_user(Vt_k, {1: 4, 2: 3}, 0)
    print("Prediction [?, 4, 3]:", pred2)


Step 1: A^T @ A =
[[66. 65.  9.]
 [65. 66.  9.]
 [ 9.  9.  2.]]

Step 3: Sorted eigenvalues: [132.24382   1.        0.75618]

Step 4: Singular values: [11.49973  1.       0.86959]

Step 5: V^T =
[[ 0.70375  0.70375  0.09726]
 [ 0.70711 -0.70711  0.     ]
 [-0.06877 -0.06877  0.99526]]

Step 6: U =
[[ 0.61197 -0.      -0.79088]
 [ 0.55923 -0.70711  0.43273]
 [ 0.55923  0.70711  0.43273]]

Step 7: Σ =
[[11.49973  0.       0.     ]
 [ 0.       1.       0.     ]
 [ 0.       0.       0.86959]]

Reconstructed matrix:
[[5. 5. 0.]
 [4. 5. 1.]
 [5. 4. 1.]]

Max reconstruction error: 0.000000
SVD valid: True
Observed V^T submatrix:
[[0.70375 0.09726]
 [0.70711 0.     ]]

User latent coordinates: [ 41.12667 -33.86062]

Prediction [5, ?, 4]: 52.88614266775167
Observed V^T submatrix:
[[ 0.70375  0.09726]
 [-0.70711  0.     ]]

User latent coordinates: [30.845   25.04191]

Prediction [?, 4, 3]: 39.41460700080852
